In [1]:
import os 
from dotenv import load_dotenv
from notion_client import Client
import json
import pandas as pd

In [2]:
load_dotenv()

True

In [3]:
notion_token = os.getenv("NOTION_TOKEN")
datasource_id = os.getenv("DATA_SOURCE_ID")
notion = Client(auth= notion_token)

In [12]:
response = notion.data_sources.query(data_source_id= datasource_id)

In [42]:
print(json.dumps(response.get('lessonObjs'), indent=2))

[
  {
    "object": "page",
    "id": "3de44339-6377-802f-8d8f-e245303a6cce",
    "created_time": "2026-09-17T17:24:00.000Z",
    "last_edited_time": "2026-09-17T17:24:00.000Z",
    "created_by": {
      "object": "user",
      "id": "3ded872b-594c-814f-b904-0002e5841bc3"
    },
    "last_edited_by": {
      "object": "user",
      "id": "3ded872b-594c-814f-b904-0002e5841bc3"
    },
    "cover": null,
    "icon": null,
    "parent": {
      "type": "data_source_id",
      "data_source_id": "f8844339-6377-8369-896d-871f5e9b1f91",
      "database_id": "17d44339-6377-83c2-98ef-8112d9f00323"
    },
    "in_trash": false,
    "is_archived": false,
    "is_locked": false,
    "properties": {
      "Date Started": {
        "id": "OQHx",
        "type": "date",
        "date": null
      },
      "Subject": {
        "id": "VhNW",
        "type": "select",
        "select": null
      },
      "Priority": {
        "id": "Xz%5De",
        "type": "select",
        "select": null
      },
    

In [22]:
print(response.get('lessonObjs')[0].keys())

dict_keys(['object', 'id', 'created_time', 'last_edited_time', 'created_by', 'last_edited_by', 'cover', 'icon', 'parent', 'in_trash', 'is_archived', 'is_locked', 'properties', 'url', 'public_url', 'archived'])


In [49]:
learnings = response.get('results')

def extract_entry(lesson):
    """Extract all properties from a Notion entry."""
    properties = lesson.get("properties", {})
    lessonObj = {}
    
    for prop_name, prop_data in properties.items():
        prop_type = prop_data.get("type")
        
        if prop_type == "date":
            date_obj = prop_data.get("date")
            lessonObj[prop_name] = date_obj.get("start") if date_obj else None
            
        elif prop_type == "select":
            select_obj = prop_data.get("select")
            lessonObj[prop_name] = select_obj.get("name") if select_obj else None
            
        elif prop_type == "status":
            status_obj = prop_data.get("status")
            lessonObj[prop_name] = status_obj.get("name") if status_obj else None
            
        elif prop_type == "url":
            lessonObj[prop_name] = prop_data.get("url")
            
        elif prop_type == "title":
            title_arr = prop_data.get("title", [])
            lessonObj[prop_name] = title_arr[0].get("plain_text") if title_arr else None
    
    return lessonObj


entries = []
for lesson in learnings:
    entries.append(extract_entry(lesson))

df = pd.DataFrame(entries)
df.head()

,Date Started,Subject,Priority,Source,Scope,Status,URL,Topic,Title
0,NaN,NaN,NaN,NaN,NaN,Not started,NaN,NaN,NaN
1,NaN,AI,Medium,Youtube,Quick Win,Not started,https://www.youtube.com/watch?v=Y5p57s7DL08&li...,Data,MCP Server on Docker
2,NaN,AI,High,Youtube,Quick Win,Not started,https://www.youtube.com/watch?v=LVzUE-v4pDw&li...,Data,LLMs & Docker
3,NaN,AI,High,Youtube,Medium,Not started,https://www.youtube.com/watch?v=vzJOAnwIokM&li...,Data,LangChain Update
4,NaN,AI,High,Youtube,Medium,Not started,https://www.youtube.com/watch?v=o126p1QN_RI&li...,Data,Crash Course Langchain
